# 02 - Sequential vs Hierarchical

CrewAI 有 3 种 process 模式:

- **sequential**:task 按列表顺序串行执行,前一个的输出自动成为后一个的 context
- **hierarchical**:增加一个 manager agent,manager 决定委托哪些 agent 做哪些 task
- consensualic(已废弃)

本 notebook 用同一个 2-agent + 2-task 的结构,对比 sequential 和
hierarchical 的输出差异。


In [ ]:
from alphaquant.infrastructure.llm import get_llm
from crewai import Agent, Task, Crew, Process

llm = get_llm(temperature=0.1)


In [ ]:
researcher = Agent(
    role="研究员",
    goal="收集信息",
    backstory="你负责收集关于主题的事实信息。",
    tools=[],
    llm=llm,
)
writer = Agent(
    role="写作者",
    goal="把研究结果写成一句简短的中文总结",
    backstory="你负责把研究发现凝练成一句话。",
    tools=[],
    llm=llm,
)


In [ ]:
research_task = Task(
    description="研究主题 'CrewAI 框架',列出 3 个关键事实。",
    expected_output="3 个关键事实的列表",
    agent=researcher,
)
write_task = Task(
    description="把研究发现写成一句简短的中文总结。",
    expected_output="一句中文总结",
    agent=writer,
# sequential 模式下,前一个 task 的输出自动成为后一个 task 的 context,
# 所以 write_task 写描述时不用显式引用 research_task。
)


In [ ]:
seq_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=False,
)
seq_result = seq_crew.kickoff(inputs={})
print("=== Sequential Result ===")
print("Research task output:")
print(seq_result.tasks_output[0].raw[:300])
print()
print("Write task output:")
print(seq_result.tasks_output[1].raw)


In [ ]:
hier_crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.hierarchical,
    manager_llm=llm,  # hierarchical 模式必须指定 manager_llm
    verbose=False,
)
hier_result = hier_crew.kickoff(inputs={})
print("=== Hierarchical Result ===")
print("Final output:")
print(hier_result.raw)


## 你看到的区别

**Sequential**:

- task 0 (research) 先跑,完整结果传给 task 1 (write)
- 你可以访问 `result.tasks_output[0]` 和 `result.tasks_output[1]` 看每一步
- 没有 manager,完全按列表顺序

**Hierarchical**:

- manager LLM 决定:这个 task 派给谁?什么时候开始?
- 你只能拿到最终 `result.raw`,中间的 task 分配对用户不可见
- manager 增加了 1 次额外的 LLM 调用(管理决策)
- 适合:task 之间依赖关系不明确、需要动态调整的场景

## 跟生产代码的对应

AlphaQuant 的 `AnalysisCrew` 用 `Process.hierarchical`,因为 8 个 task 之间的
依赖是隐式的(数据收集 → 分析 → 写报告),manager 能更好地协调。
在 `crews/analysis_crew.py` 的 `_build_crew()` 里可以看到具体配置。
